# CSC2042S Assignment 1
## Perceptron Image Classification

**Student:** Liso Njena  
**Student number:** NJNLIS001

This notebook implements a multi-class perceptron from scratch and applies it to the Simpsons-MNIST dataset. Explanations are included before each code section so that the design and results can be understood and reproduced.

# Task 1: Data processing

The perceptron cannot work directly with JPEG files. The images must first be loaded into NumPy arrays and paired with numeric class labels.

This section:

1. Loads the grayscale and RGB training and test images with Pillow.
2. Preserves the raw, unflattened pixel arrays for later data augmentation.
3. Splits only the supplied training set into training and validation subsets.
4. Uses the same split for RGB and grayscale images so that the comparison is fair.
5. Flattens images into feature vectors and provides the three normalisation options required later.

The supplied test set is kept separate. It must not be used to select hyperparameters.

## 1.1 Imports and reproducibility

- `Path` constructs file paths that work across operating systems.
- NumPy stores images and performs numerical calculations.
- Pillow opens and converts the JPEG images.
- Matplotlib displays samples for a visual data check.
- `train_test_split` creates a stratified training/validation split.
- A fixed random seed ensures that the same split can be reproduced.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

## 1.2 Class labels

The dataset stores each character in a separate folder. A perceptron requires numeric targets, so every folder name is mapped to one label from 0 to 9. The ordering below follows the dataset's documented label mapping.

In [ ]:
CLASS_NAMES = [
    "bart_simpson",
    "charles_montgomery_burns",
    "homer_simpson",
    "krusty_the_clown",
    "lisa_simpson",
    "marge_simpson",
    "milhouse_van_houten",
    "moe_szyslak",
    "ned_flanders",
    "principal_skinner",
]

CLASS_TO_LABEL = {
    class_name: label
    for label, class_name in enumerate(CLASS_NAMES)
}

LABEL_TO_CLASS = {
    label: class_name
    for class_name, label in CLASS_TO_LABEL.items()
}

CLASS_TO_LABEL

## 1.3 Loading JPEG images

The loader visits the ten character folders in a fixed order and sorts the filenames. Sorting makes the loading order reproducible and keeps corresponding RGB and grayscale images aligned.

Pillow mode `"L"` produces a grayscale array of shape `(28, 28)`. Mode `"RGB"` produces an RGB array of shape `(28, 28, 3)`.

Raw pixels are stored as `np.uint8`, whose values range from 0 to 255. The images are not flattened or normalised inside this function because the raw image shape is required for the augmentation experiments in Task 7.

In [ ]:
def load_dataset(split_directory, image_mode):
    """Load one train/test directory into raw image and label arrays.

    Parameters
    ----------
    split_directory : str or Path
        Directory containing one subfolder per Simpsons character.
    image_mode : {"L", "RGB"}
        Pillow colour mode. "L" loads grayscale; "RGB" loads colour.

    Returns
    -------
    images : np.ndarray
        Raw uint8 images. Shape is (N, 28, 28) for grayscale or
        (N, 28, 28, 3) for RGB.
    labels : np.ndarray
        Integer class labels with shape (N,).
    image_ids : list[str]
        Relative identifiers used to verify RGB/grayscale alignment.
    """
    split_directory = Path(split_directory)

    if image_mode == "L":
        expected_shape = (28, 28)
    elif image_mode == "RGB":
        expected_shape = (28, 28, 3)
    else:
        raise ValueError("image_mode must be either 'L' or 'RGB'")

    images = []
    labels = []
    image_ids = []

    for label, class_name in enumerate(CLASS_NAMES):
        class_directory = split_directory / class_name

        if not class_directory.is_dir():
            raise FileNotFoundError(
                f"Class folder not found: {class_directory}"
            )

        image_paths = sorted(
            path
            for path in class_directory.iterdir()
            if path.suffix.lower() in {".jpg", ".jpeg"}
        )

        if not image_paths:
            raise FileNotFoundError(
                f"No JPEG images found in: {class_directory}"
            )

        for image_path in image_paths:
            # The context manager closes the image file after it is read.
            with Image.open(image_path) as image:
                image_array = np.asarray(
                    image.convert(image_mode),
                    dtype=np.uint8,
                )

            if image_array.shape != expected_shape:
                raise ValueError(
                    f"{image_path} has shape {image_array.shape}; "
                    f"expected {expected_shape}"
                )

            images.append(image_array)
            labels.append(label)
            image_ids.append(f"{class_name}/{image_path.name}")

    return (
        np.stack(images),
        np.asarray(labels, dtype=np.int64),
        image_ids,
    )

## 1.4 Locating the dataset

The local dataset folder is named `A2-dataset` and is deliberately ignored by Git because the assignment says not to submit the data.

The second condition below also supports the structure obtained by cloning the original Simpsons-MNIST repository, where the `grayscale` and `rgb` folders are nested inside another folder named `dataset`.

In [ ]:
DATA_ROOT = Path("A2-dataset")

# Support A2-dataset/dataset/grayscale/... as well as
# A2-dataset/grayscale/...
if (DATA_ROOT / "dataset").is_dir():
    DATA_ROOT = DATA_ROOT / "dataset"

required_directories = [
    DATA_ROOT / "grayscale" / "train",
    DATA_ROOT / "grayscale" / "test",
    DATA_ROOT / "rgb" / "train",
    DATA_ROOT / "rgb" / "test",
]

missing_directories = [
    path for path in required_directories
    if not path.is_dir()
]

if missing_directories:
    missing_text = "\n".join(str(path) for path in missing_directories)
    raise FileNotFoundError(
        "The following dataset directories were not found:\n"
        f"{missing_text}\n\n"
        "Check that A2-dataset is extracted and that the notebook "
        "is being run from the repository folder."
    )

print("Working directory:", Path.cwd())
print("Dataset directory:", DATA_ROOT.resolve())

## 1.5 Loading the four supplied sets

The supplied training images are called the *full training set* here because they will shortly be divided into a smaller training set and a held-out validation set.

The supplied test set is loaded but will remain untouched during model development and hyperparameter tuning.

In [ ]:
X_gray_full_raw, y_gray_full, gray_train_ids = load_dataset(
    DATA_ROOT / "grayscale" / "train",
    image_mode="L",
)

X_rgb_full_raw, y_rgb_full, rgb_train_ids = load_dataset(
    DATA_ROOT / "rgb" / "train",
    image_mode="RGB",
)

X_gray_test_raw, y_gray_test, gray_test_ids = load_dataset(
    DATA_ROOT / "grayscale" / "test",
    image_mode="L",
)

X_rgb_test_raw, y_rgb_test, rgb_test_ids = load_dataset(
    DATA_ROOT / "rgb" / "test",
    image_mode="RGB",
)

print("All four datasets loaded successfully.")

## 1.6 Verifying the loaded data

These assertions are deliberate correctness checks:

- Both modalities must contain corresponding images in the same order.
- Their labels must agree.
- Grayscale and RGB arrays must have the expected dimensions.
- The official dataset should contain 8,000 supplied training images and 2,000 test images.

If an assertion fails, the data-loading problem should be fixed before training a model.

In [ ]:
assert gray_train_ids == rgb_train_ids
assert gray_test_ids == rgb_test_ids
assert np.array_equal(y_gray_full, y_rgb_full)
assert np.array_equal(y_gray_test, y_rgb_test)

assert X_gray_full_raw.shape == (8000, 28, 28)
assert X_rgb_full_raw.shape == (8000, 28, 28, 3)
assert X_gray_test_raw.shape == (2000, 28, 28)
assert X_rgb_test_raw.shape == (2000, 28, 28, 3)

print("Grayscale supplied training:", X_gray_full_raw.shape)
print("RGB supplied training:      ", X_rgb_full_raw.shape)
print("Grayscale test:             ", X_gray_test_raw.shape)
print("RGB test:                   ", X_rgb_test_raw.shape)
print("Raw pixel data type:        ", X_rgb_full_raw.dtype)
print("Raw pixel range:            ",
      X_rgb_full_raw.min(), "to", X_rgb_full_raw.max())

## 1.7 Visual inspection

A visual check helps catch incorrect labels, colour modes, or corrupted images. The following cell displays one RGB example from each class.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for label, axis in enumerate(axes.flat):
    example_index = np.flatnonzero(y_rgb_full == label)[0]
    axis.imshow(X_rgb_full_raw[example_index])
    axis.set_title(CLASS_NAMES[label].replace("_", " "))
    axis.axis("off")

fig.suptitle("One RGB training example from each class", fontsize=14)
plt.tight_layout()
plt.show()

## 1.8 Training/validation split

The supplied training set is divided into 80% training data and 20% validation data:

- Training: 6,400 images
- Validation: 1,600 images
- Test: the original 2,000 images

A **stratified** split preserves the proportion of all ten classes. The split is performed on indices and the same indices are applied to RGB and grayscale arrays. This ensures that both modalities are trained and validated on corresponding examples.

The validation set will guide hyperparameter selection. The test set must only be used for the final evaluation.

In [ ]:
all_indices = np.arange(y_gray_full.shape[0])

train_indices, validation_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y_gray_full,
)

# Grayscale split
X_gray_train_raw = X_gray_full_raw[train_indices]
X_gray_validation_raw = X_gray_full_raw[validation_indices]

# RGB split using exactly the same image indices
X_rgb_train_raw = X_rgb_full_raw[train_indices]
X_rgb_validation_raw = X_rgb_full_raw[validation_indices]

# Labels are shared because RGB and grayscale images are aligned.
y_train = y_gray_full[train_indices]
y_validation = y_gray_full[validation_indices]
y_test = y_gray_test.copy()

print("Training labels:  ", y_train.shape)
print("Validation labels:", y_validation.shape)
print("Test labels:      ", y_test.shape)

## 1.9 Checking class balance

The dataset is balanced, and stratification should preserve this balance. Printing the counts verifies that each character is properly represented in every split.

In [ ]:
def class_counts(labels):
    """Return the number of examples belonging to each class."""
    return np.bincount(labels, minlength=len(CLASS_NAMES))


train_counts = class_counts(y_train)
validation_counts = class_counts(y_validation)
test_counts = class_counts(y_test)

print(f"{'Class':32s} {'Train':>7s} {'Validation':>12s} {'Test':>7s}")
print("-" * 62)

for label, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:32s} "
        f"{train_counts[label]:7d} "
        f"{validation_counts[label]:12d} "
        f"{test_counts[label]:7d}"
    )

## 1.10 Flattening and normalisation

A perceptron expects each example to be a one-dimensional feature vector.

- Grayscale: `(28, 28) -> (784,)`
- RGB: `(28, 28, 3) -> (2352,)`

The function below supports the three preprocessing conditions required for hyperparameter tuning:

- `"none"`: retain pixel values from 0 to 255.
- `"zero_one"`: divide pixels by 255 to obtain values from 0 to 1.
- `"z_score"`: subtract the training mean and divide by the training standard deviation.

For z-score normalisation, statistics are calculated **only from the training set** and then applied to validation and test data. Calculating them using validation or test data would cause data leakage.

In [ ]:
def preprocess_splits(
    train_images,
    validation_images,
    test_images,
    method="none",
):
    """Flatten and consistently normalise train/validation/test images."""
    # Convert uint8 pixels to float32 before arithmetic.
    train = train_images.reshape(train_images.shape[0], -1).astype(np.float32)
    validation = validation_images.reshape(
        validation_images.shape[0], -1
    ).astype(np.float32)
    test = test_images.reshape(test_images.shape[0], -1).astype(np.float32)

    if method == "none":
        return train, validation, test

    if method == "zero_one":
        return train / 255.0, validation / 255.0, test / 255.0

    if method == "z_score":
        # Each flattened pixel position is treated as one feature.
        feature_mean = train.mean(axis=0, keepdims=True)
        feature_std = train.std(axis=0, keepdims=True)

        # Constant features have standard deviation zero. Dividing them
        # by 1 instead prevents division-by-zero without changing them.
        safe_std = np.where(feature_std < 1e-8, 1.0, feature_std)

        return (
            (train - feature_mean) / safe_std,
            (validation - feature_mean) / safe_std,
            (test - feature_mean) / safe_std,
        )

    raise ValueError(
        "method must be 'none', 'zero_one', or 'z_score'"
    )

## 1.11 Preparing the initial feature arrays

For the initial implementation, no normalisation is applied. Task 4 will systematically compare this baseline with 0-to-1 scaling and z-score normalisation.

The expected final feature shapes are:

- Grayscale training: `(6400, 784)`
- RGB training: `(6400, 2352)`

In [ ]:
X_gray_train, X_gray_validation, X_gray_test = preprocess_splits(
    X_gray_train_raw,
    X_gray_validation_raw,
    X_gray_test_raw,
    method="none",
)

X_rgb_train, X_rgb_validation, X_rgb_test = preprocess_splits(
    X_rgb_train_raw,
    X_rgb_validation_raw,
    X_rgb_test_raw,
    method="none",
)

print("Grayscale features:")
print("  Train:     ", X_gray_train.shape)
print("  Validation:", X_gray_validation.shape)
print("  Test:      ", X_gray_test.shape)

print("\nRGB features:")
print("  Train:     ", X_rgb_train.shape)
print("  Validation:", X_rgb_validation.shape)
print("  Test:      ", X_rgb_test.shape)

## Task 1 summary

The JPEG files have now been transformed into data suitable for a perceptron:

- All images have numeric labels from 0 to 9.
- Raw arrays are retained for the later augmentation task.
- The supplied training set is split reproducibly and stratified by class.
- RGB and grayscale models use corresponding train/validation examples.
- Each image can be flattened and processed using any required normalisation method.
- Training-derived statistics are used to avoid validation/test leakage.
- The test set remains reserved for final evaluation.